# Europeana: an aggregator for European cultural heritage

The previous notebook showed what IIIF makes possible *within one platform*: a manifest describes an object, and an image service delivers its pixels. That raises a practical question. There are thousands of libraries, archives and museums, each with its own platform. How does a researcher find objects across all of them?

The usual answer is an **aggregator**: a service that collects descriptions of objects from many institutions into one searchable index. [Europeana](https://www.europeana.eu/) is the largest for European cultural heritage. In this notebook you will:

1. See how Europeana collects its data, and where IIIF fits in.
2. Use the Europeana **Search API** and read its response.
3. Build an overview of the contributing **institutions**: item counts by type (image, text, sound, video, 3D) and a preview of what a single institution holds.
4. Ask questions the Europeana website's filters do not answer directly: counts across fields, distributions over time, combined filters, and an export of metadata as a dataset.
5. Look at what a single record contains, including the link to its IIIF manifest.

## 1. How the data gets to Europeana

![Europeana flow: institutions reach Europeana directly, through aggregators, or through projects](../misc/assets/img/europeana_flow.svg)

The diagram, from the bottom up:

- Institutions run their own **platforms** (for example e-codices or e-rara for Swiss manuscripts and rare prints). A platform exposes its data in two separate ways:
  - **Metadata** (title, creator, date, rights, …) through a harvesting protocol, typically **OAI-PMH**, in formats such as METS/MODS or Dublin Core.
  - **Images and manifests** through **IIIF**, as in the previous notebook. Viewers and research tools read these directly.
- Europeana **harvests the metadata** and converts it to its own format, the Europeana Data Model (**EDM**). Institutions reach Europeana by one of three routes:
  - *directly*, for large providers;
  - *via an aggregator*, a national or domain-specific one such as Germany's Deutsche Digitale Bibliothek (DDB);
  - *via a project*, for thematic EU-funded projects.
- Where an institution also offers IIIF, the Europeana record can link to its manifest (the dashed line).

Two consequences are worth keeping in mind through the rest of the notebook:

- Europeana is mainly an **index of descriptions**. A record holds metadata, a small preview image, and links back to the institution's own page and media file. The full-resolution objects stay with the institutions.
- Europeana records carry two names: the **data provider** (the institution that holds the object) and the **provider** (the aggregator or route that delivered it). The API lets you count by either, and the tables below use both.

## 2. The Europeana API

An **API** (Application Programming Interface) lets a program send a request to a server and receive structured data, usually JSON, instead of a web page. Europeana offers several. This notebook uses two:

| API | Purpose | Example |
|---|---|---|
| **Search API** | Query and filter the whole index; also returns counts per value ("facets") | `…/record/v2/search.json?query=river` |
| **Record API** | Full metadata of one item | `…/record/v2/{dataset}/{local_id}.json` |

Requests carry a free API key (`wskey`). Useful parameters of the Search API:

| Parameter | Meaning | Example |
|---|---|---|
| `query` | search text (`*` for everything) | `Rembrandt` |
| `qf` | filter on a field | `TYPE:IMAGE`, `COUNTRY:Sweden` |
| `theme` | a thematic collection | `art`, `manuscripts`, `photography` |
| `rows` | number of results (max 100) | `12` |
| `profile` | how much to return (`standard`, `rich`, `facets`) | `facets` |
| `facet` | count items per value of a field | `DATA_PROVIDER` |

## Setup

In [ ]:
# Standard library imports
import html
import json
import time
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# External libraries
import requests
from IPython.display import HTML, display, Image

# Set up project paths
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

IMAGES_DIR = PROJECT_ROOT / "data" / "images"
DATA_DIR   = PROJECT_ROOT / "data"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Images dir:   {IMAGES_DIR}")
print(f"Data dir:     {DATA_DIR}")

### API key

Create a free key at <https://pro.europeana.eu/page/get-api> and save it to `notebooks/api-key-europeana.txt`. Without one the notebook falls back to a shared demo key, which is enough for everything here but is rate limited.

In [ ]:
# Load API key - check multiple locations
NOTEBOOK_DIR = Path(".").resolve()
API_KEY_LOCATIONS = [
    NOTEBOOK_DIR / "api-key-europeana.txt",
    PROJECT_ROOT / "misc" / "api-key-europeana.txt",
    PROJECT_ROOT / "api-key-europeana.txt"
]

# Default demo key (limited requests)
API_KEY = "api2demo"

for key_file in API_KEY_LOCATIONS:
    if key_file.exists():
        with open(key_file, 'r') as f:
            custom_key = f.read().strip()
            if custom_key and custom_key != "api2demo":
                API_KEY = custom_key
                print(f"✓ API key loaded from {key_file}")
                break
else:
    print(f"ℹ Using demo API key (limited to 999 requests)")
    print(f"  For unlimited access, get your own key at: https://pro.europeana.eu/page/get-api")
    print(f"  Save it to: notebooks/api-key-europeana.txt")

# Base API endpoint
BASE_URL = "https://api.europeana.eu/record/v2"
print(f"\nAPI endpoint: {BASE_URL}")

## 3. An overview of the contributing institutions

The Search API can count items per value of a field. Setting `rows=0` returns no items at all, only the counts. Four fields are relevant here:

| Facet | Meaning |
|---|---|
| `DATA_PROVIDER` | the institution that holds the object |
| `PROVIDER` | the aggregator or route through which it reached Europeana |
| `COUNTRY` | the country of the data provider |
| `TYPE` | the media type of the item: `IMAGE`, `TEXT`, `SOUND`, `VIDEO`, `3D` |

Europeana's `TYPE` values describe the media, not the genre. In practice `TEXT` is where books, manuscripts, newspapers and archival documents sit, and `IMAGE` holds paintings, photographs, maps and prints. "Art" is not a type but a **thematic collection** (`theme=art`) that cuts across types, so it is counted separately and overlaps with the other columns.

A first look at the whole index:

In [ ]:
def search_raw(**params):
    r = requests.get(f"{BASE_URL}/search.json", params={"wskey": API_KEY, **params}, timeout=60)
    r.raise_for_status()
    return r.json()

def facet_counts(facet, page=2000, **extra):
    # Return every (value, count) pair of a facet, following the facet offset until it runs out.
    out, offset = [], 0
    while True:
        data = search_raw(query="*", rows=0, profile="facets", facet=facet,
                          **{f"f.{facet}.facet.limit": page, f"f.{facet}.facet.offset": offset}, **extra)
        fields = next((f["fields"] for f in data.get("facets", []) if f["name"] == facet), [])
        out += [(f["label"], f["count"]) for f in fields]
        if len(fields) < page:
            return out
        offset += page

TYPES = ["IMAGE", "TEXT", "SOUND", "VIDEO", "3D"]

total_items = search_raw(query="*", rows=0)["totalResults"]
print(f"Items in the index: {total_items:,}\n")
for t, n in facet_counts("TYPE"):
    print(f"  {t:<6} {n:>12,}   {n / total_items:6.1%}")

### Collecting the counts once

To split every institution by type, the notebook asks for the `DATA_PROVIDER` counts once per type (with `qf=TYPE:…`), and once more for the Art theme. The same is done for aggregators (`PROVIDER`) and countries. That is about 35 small requests in total. A further column counts the records that link to a IIIF manifest (`wr_dcterms_isReferencedBy:*`), which tells us which institutions can serve images through IIIF. The results are saved to `data/europeana/overview_cache.json`, so later runs read the file instead of calling the API. Set `REFRESH = True` to fetch fresh numbers.

In [ ]:
REFRESH = False
CACHE = DATA_DIR / "europeana" / "overview_cache.json"
CACHE.parent.mkdir(parents=True, exist_ok=True)

def collect_overview():
    result = {"fetched": time.strftime("%Y-%m-%d"), "total_items": total_items, "facets": {}}
    for facet in ["DATA_PROVIDER", "PROVIDER", "COUNTRY"]:
        columns = {"total": facet_counts(facet)}
        for t in TYPES:
            columns[t] = facet_counts(facet, qf=f"TYPE:{t}")
        if facet == "DATA_PROVIDER":
            columns["art"] = facet_counts(facet, theme="art")
            columns["iiif"] = facet_counts(facet, qf="wr_dcterms_isReferencedBy:*")
        result["facets"][facet] = {k: dict(v) for k, v in columns.items()}
        print(f"  {facet}: {len(columns['total']):,} values")
    return result

if CACHE.exists() and not REFRESH:
    overview = json.loads(CACHE.read_text())
    print(f"Loaded cached counts from {overview['fetched']}")
    if "iiif" not in overview["facets"]["DATA_PROVIDER"]:      # cache written before this column existed
        overview["facets"]["DATA_PROVIDER"]["iiif"] = dict(facet_counts("DATA_PROVIDER", qf="wr_dcterms_isReferencedBy:*"))
        CACHE.write_text(json.dumps(overview))
else:
    print("Fetching counts from the API ...")
    overview = collect_overview()
    CACHE.write_text(json.dumps(overview))
    print("Saved to", CACHE)

def table_rows(facet):
    # One dict per value of the facet, with a count for the total, each type and (for institutions) the Art theme.
    cols = overview["facets"][facet]
    return [{"name": name, **{c: cols[c].get(name, 0) for c in cols}} for name in cols["total"]]

institutions = table_rows("DATA_PROVIDER")
aggregators  = table_rows("PROVIDER")
countries    = table_rows("COUNTRY")
print(f"\n{len(institutions):,} institutions (data providers), {len(aggregators)} providers / aggregators, {len(countries)} countries")

### Institutions

The table lists the institutions with the most items. The coloured bar shows each institution's mix of media types as a share of its own total, so a small institution and a large one can be compared. The exact counts are in the columns.

In [ ]:
COLORS = {"IMAGE": "#2a78d6", "TEXT": "#eb6834", "SOUND": "#1baf7a", "VIDEO": "#eda100", "3D": "#e87ba4"}
LABELS = {"IMAGE": "Image", "TEXT": "Text", "SOUND": "Sound", "VIDEO": "Video", "3D": "3D"}

def mix_bar(row, width=170):
    typed = sum(row[t] for t in TYPES) or 1
    parts = "".join(
        f"<span title='{LABELS[t]}: {row[t]:,} ({row[t] / typed:.0%})' "
        f"style='display:inline-block;height:12px;width:{max(row[t] / typed * width, 0):.1f}px;background:{COLORS[t]};margin-right:{1 if row[t] else 0}px'></span>"
        for t in TYPES)
    return f"<div style='white-space:nowrap;line-height:0'>{parts}</div>"

def show_table(rows, sort_by="total", n=20, first_col="Institution", art=False, iiif=False, min_total=0):
    rows = [r for r in rows if r["total"] >= min_total]
    rows = sorted(rows, key=lambda r: r[sort_by], reverse=True)[:n]
    head = (f"<th style='text-align:left'>{first_col}</th><th>Total</th>"
            + "".join(f"<th>{LABELS[t]}</th>" for t in TYPES) + ("<th>Art theme</th>" if art else "") + ("<th>IIIF-linked</th>" if iiif else "") + "<th style='text-align:left'>Media mix</th>")
    body = ""
    for r in rows:
        cells = "".join(f"<td style='text-align:right'>{r[t]:,}</td>" if r[t] else "<td style='text-align:right;color:#999'>–</td>" for t in TYPES)
        body += (f"<tr><td>{html.escape(r['name'])}</td><td style='text-align:right'><b>{r['total']:,}</b></td>{cells}"
                 + (f"<td style='text-align:right'>{r['art']:,}</td>" if art else "")
                 + (f"<td style='text-align:right'>{r['iiif']:,}</td>" if iiif else "") + f"<td>{mix_bar(r)}</td></tr>")
    legend = " &nbsp; ".join(f"<span style='display:inline-block;width:10px;height:10px;background:{COLORS[t]}'></span> {LABELS[t]}" for t in TYPES)
    display(HTML(f"<div style='font:13px sans-serif;margin-bottom:6px'>{legend}</div>"
                 f"<table style='font:13px sans-serif'><tr>{head}</tr>{body}</table>"))

show_table(institutions, sort_by="total", n=20, art=True, iiif=True)

Because `TYPE`, the Art theme and the IIIF column are counted separately, the type columns add up to the total (an item has exactly one type), while the Art and IIIF-linked columns are subsets that overlap them. "IIIF-linked" counts records whose metadata name a IIIF manifest (see section 5). It is a **lower bound**: an institution can serve images through IIIF without naming the manifest in its Europeana records, and then shows 0 here (the Wellcome Collection is an example, as section 5 shows).

The same table sorted differently answers other questions. Which institutions contribute **3D objects**?

In [ ]:
show_table(institutions, sort_by="3D", n=15, art=True, iiif=True)

And which hold the most **text** (books, manuscripts, newspapers)?

In [ ]:
show_table(institutions, sort_by="TEXT", n=15, art=True, iiif=True)

Contributions are very uneven across institutions:

In [ ]:
totals = sorted((r["total"] for r in institutions), reverse=True)
def share(k): return sum(totals[:k]) / sum(totals)
print(f"Institutions with at least 1 item:        {len(totals):,}")
print(f"Median institution:                       {totals[len(totals)//2]:,} items")
print(f"Top 10 institutions hold                  {share(10):.0%} of all items")
print(f"Top 100 institutions hold                 {share(100):.0%} of all items")
print(f"Institutions with fewer than 1,000 items: {sum(t < 1000 for t in totals):,}")

Which institutions name the most IIIF manifests in their records? For these, the techniques from the first notebook (small versions, regions, deep zoom) can be applied directly. The list is certainly incomplete, for the reason given above:

In [ ]:
iiif_top = sorted((r for r in institutions if r["iiif"]), key=lambda r: r["iiif"], reverse=True)
print(f"{len(iiif_top):,} of {len(institutions):,} institutions have at least one record linking a IIIF manifest")
print(f"{sum(r['iiif'] for r in institutions):,} of {total_items:,} records ({sum(r['iiif'] for r in institutions) / total_items:.1%})\n")
show_table(institutions, sort_by="iiif", n=15, iiif=True)

### Providers and aggregators

The `PROVIDER` facet shows the route by which items arrived, matching the middle row of the diagram. Names such as the German Digital Library (DDB) or Hispana (Spain) are national aggregators. "OpenUp!" is a project collecting natural history material.

In [ ]:
show_table(aggregators, sort_by="total", n=20, first_col="Provider / aggregator")

### Countries

In [ ]:
show_table(countries, sort_by="total", n=20, first_col="Country")

The counts depend on how generously each country and aggregator delivers data, so they describe what has been contributed to Europeana, not the size of a country's heritage.

To keep the numbers for slides or further analysis, the tables can be written to CSV:

In [ ]:
import csv

def write_csv(rows, name):
    path = DATA_DIR / "europeana" / name
    cols = ["name", "total"] + TYPES + (["art", "iiif"] if "art" in rows[0] else [])
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(sorted(rows, key=lambda r: r["total"], reverse=True))
    return path

for rows, name in [(institutions, "overview_institutions.csv"), (aggregators, "overview_providers.csv"), (countries, "overview_countries.csv")]:
    print("wrote", write_csv(rows, name).relative_to(PROJECT_ROOT))

A few helper functions for searching and for reading common fields from a result:

In [ ]:
def search_europeana(query="*", rows=12, reusability="open", qf=None, profile="rich", cursor=None, theme=None):
    """
    Search the Europeana collection.

    Parameters:
        query: Search term (default: "*" for all)
        rows: Number of results to return (max 100)
        reusability: Filter by license ("open", "restricted", "permission", or None)
        qf: Additional query filters as list (e.g., ["TYPE:IMAGE", "COUNTRY:Netherlands"])
        profile: "standard", "rich", or "facets" for more metadata
        cursor: Cursor for pagination (use for getting more than 100 results)
        theme: Thematic collection filter (e.g., "art", "fashion", "music", "photography",
               "nature", "newspapers", "sport", "ww1", "archaeology", "migration",
               "maps-and-geography", "manuscripts") or None for all

    Returns:
        Dictionary with search results
    """
    url = f"{BASE_URL}/search.json"

    params = {
        "wskey": API_KEY,
        "query": query,
        "rows": min(rows, 100),
        "profile": profile
    }

    if reusability:
        params["reusability"] = reusability

    if qf:
        params["qf"] = qf

    if cursor:
        params["cursor"] = cursor

    if theme:
        params["theme"] = theme

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error searching Europeana: {e}")
        return None

In [ ]:
# Helper functions for extracting metadata

def get_item_title(item):
    """Extract title from an item."""
    if 'title' in item and item['title']:
        return item['title'][0] if isinstance(item['title'], list) else item['title']
    if 'dcTitleLangAware' in item:
        for lang in ['en', 'nl', 'de', 'fr', 'sv', 'def']:
            if lang in item['dcTitleLangAware']:
                return item['dcTitleLangAware'][lang][0]
    return "Untitled"


def get_item_creator(item):
    """Extract creator/artist from an item."""
    if 'dcCreator' in item and item['dcCreator']:
        return item['dcCreator'][0] if isinstance(item['dcCreator'], list) else item['dcCreator']
    return "Unknown"


def get_item_year(item):
    """Extract year from an item."""
    if 'year' in item and item['year']:
        return item['year'][0] if isinstance(item['year'], list) else item['year']
    return "n.d."


def get_item_preview(item):
    """Extract preview image URL."""
    if 'edmPreview' in item and item['edmPreview']:
        return item['edmPreview'][0] if isinstance(item['edmPreview'], list) else item['edmPreview']
    return None


def get_full_image_url(item):
    """Extract full-size image URL (from source institution)."""
    if 'edmIsShownBy' in item and item['edmIsShownBy']:
        return item['edmIsShownBy'][0] if isinstance(item['edmIsShownBy'], list) else item['edmIsShownBy']
    return None


def sanitize_filename(name):
    """Remove problematic characters from filenames."""
    if not name:
        return "unknown"
    safe = "".join(c for c in name if c.isalnum() or c in ' ._-')
    return safe.strip()[:80]

### A closer look at one institution

The table says how much an institution holds, not what. A few more facet requests, restricted to one institution with `qf=DATA_PROVIDER:"name"`, give a quick profile: media types, rights statements, the period covered, the most frequent creators, and a handful of sample images.

The samples are taken from a random position within the first 1,000 results (the API does not page deeper with `start`), so they come from the front of the institution's result list in Europeana's default order, not from a true random draw.

In [ ]:
import random

def century_label(c):
    suffix = "th" if 10 <= c % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(c % 10, "th")
    return f"{c}{suffix} c."

def bar_table(pairs, total, title, label_fmt=lambda x: x):
    rows = "".join(
        f"<tr><td style='padding-right:10px'>{html.escape(str(label_fmt(k)))}</td><td style='text-align:right;padding-right:8px'>{v:,}</td>"
        f"<td><div style='background:#2a78d6;height:10px;width:{max(v / total * 160, 1):.0f}px'></div></td></tr>" for k, v in pairs)
    return f"<div><b>{title}</b><table style='font:12px sans-serif'>{rows}</table></div>"

def institution_preview(name, n_samples=12):
    flt = f'DATA_PROVIDER:"{name}"'
    data = search_raw(query="*", rows=0, qf=flt, profile="facets", facet="TYPE,RIGHTS,YEAR,proxy_dc_creator",
                      **{"f.TYPE.facet.limit": 6, "f.RIGHTS.facet.limit": 5, "f.YEAR.facet.limit": 2000, "f.proxy_dc_creator.facet.limit": 8})
    total = data["totalResults"]
    fac = {f["name"]: [(x["label"], x["count"]) for x in f["fields"]] for f in data.get("facets", [])}

    centuries = {}
    for y, c in fac.get("YEAR", []):
        if y.isdigit():
            centuries[int(y) // 100 + 1] = centuries.get(int(y) // 100 + 1, 0) + c
    rights = lambda u: u.rstrip("/").split("/")[-2] + " " + u.rstrip("/").split("/")[-1] if "creativecommons" in u or "rightsstatements" in u else u
    has_images = any(t == "IMAGE" for t, _ in fac.get("TYPE", []))

    start = random.randint(1, max(1, min(total, 1000) - n_samples + 1))
    sample_filter = [flt, "TYPE:IMAGE"] if has_images else [flt]
    sample = search_raw(query="*", rows=n_samples, start=start, qf=sample_filter, profile="standard").get("items", [])

    panels = "".join([
        bar_table(fac.get("TYPE", []), total, "Media types"),
        bar_table(fac.get("RIGHTS", []), total, "Rights (top 5)", rights),
        bar_table(sorted(centuries.items()), max(centuries.values(), default=1), "Items by century (dated items)", century_label),
        bar_table(fac.get("proxy_dc_creator", []), max((c for _, c in fac.get("proxy_dc_creator", [])), default=1), "Most frequent creators"),
    ])
    thumbs = "".join(
        f"<figure style='margin:0;width:150px;font:11px sans-serif'><img src='{(it.get('edmPreview') or [''])[0]}' style='width:150px;height:150px;object-fit:cover;background:#eee'>"
        f"<figcaption>{html.escape(get_item_title(it)[:70])}</figcaption></figure>" for it in sample)
    display(HTML(f"<h4 style='font-family:sans-serif'>{html.escape(name)}: {total:,} items</h4>"
                 f"<div style='display:flex;gap:28px;flex-wrap:wrap'>{panels}</div>"
                 f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin-top:12px'>{thumbs}</div>"))

Try it with an institution from the tables above. Name it exactly as it appears there:

In [ ]:
INSTITUTION = "Wellcome Collection"      # <-- change this, e.g. "Deutsche Fotothek", "Rijksmuseum", "Naturalis Biodiversity Center"

institution_preview(INSTITUTION)

## 4. Beyond the website's search box

The Europeana website already offers keyword search with filters, and it does that well. The API is more useful for questions that need *numbers across the whole index*, *combinations of fields*, or *the metadata as a table*. This section shows four of these.

### 4.1 Distributions: how are results spread over time, institutions and rights?

A facet request returns counts per value of a field for any query. The website shows facets as a sidebar for the current search. Through the API they can be requested for any field and collected into a table or a chart. For example, for the query `river`:

In [ ]:
def facet_pairs(facet, query="*", limit=10, qf=None, **extra):
    data = search_raw(query=query, rows=0, profile="facets", facet=facet, qf=qf, **{f"f.{facet}.facet.limit": limit}, **extra)
    fields = next((f["fields"] for f in data.get("facets", []) if f["name"] == facet), [])
    return data["totalResults"], [(x["label"], x["count"]) for x in fields]

QUERY = "river"
total_river, by_institution = facet_pairs("DATA_PROVIDER", QUERY, limit=8)
_, years = facet_pairs("YEAR", QUERY, limit=2000)

by_century = {}
for y, c in years:
    if y.isdigit() and 1000 <= int(y) <= 2025:
        by_century[int(y) // 100 + 1] = by_century.get(int(y) // 100 + 1, 0) + c

display(HTML(
    f"<h4 style='font-family:sans-serif'>'{QUERY}': {total_river:,} items</h4><div style='display:flex;gap:40px;flex-wrap:wrap'>"
    + bar_table(by_institution, by_institution[0][1], "Institutions with the most matches")
    + bar_table(sorted(by_century.items()), max(by_century.values()), "Dated items by century (11th to 21st)", century_label)
    + "</div>"))

A cross-tabulation combines two fields, for example rights statements per media type. It takes one request per row:

In [ ]:
def short_rights(u):
    return u.rstrip("/").split("/")[-2] + " " + u.rstrip("/").split("/")[-1] if ("creativecommons" in u or "rightsstatements" in u) else u

table = {}
for t in ["IMAGE", "TEXT", "3D"]:
    n, pairs = facet_pairs("RIGHTS", "*", limit=6, qf=f"TYPE:{t}")
    table[t] = [(short_rights(k), v / n) for k, v in pairs]

display(HTML("<div style='display:flex;gap:40px;flex-wrap:wrap;font-family:sans-serif'>" + "".join(
    "<div><b>" + t + "</b><table style='font:12px sans-serif'>"
    + "".join(f"<tr><td style='padding-right:10px'>{html.escape(k)}</td><td style='text-align:right'>{v:.0%}</td></tr>" for k, v in rows)
    + "</table></div>" for t, rows in table.items()) + "</div>"))

### 4.2 Combined field and range filters

Filters can be stacked, and some are not available as simple checkboxes on the website: ranges on a field (`YEAR:[1800 TO 1899]`), wildcards, and the IIIF filter used earlier. Each step below narrows the previous one:

In [ ]:
def count(query="*", **kw):
    return search_raw(query=query, rows=0, **kw)["totalResults"]

steps = [
    ("query: ship",                                   dict(query="ship")),
    ("+ images only",                                 dict(query="ship", qf=["TYPE:IMAGE"])),
    ("+ openly reusable",                             dict(query="ship", qf=["TYPE:IMAGE"], reusability="open")),
    ("+ dated 1800 to 1899",                          dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]"], reusability="open")),
    ("+ record names a IIIF manifest (lower bound)", dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]", "wr_dcterms_isReferencedBy:*"], reusability="open")),
]
for label, kw in steps:
    print(f"{label:<44} {count(**kw):>10,}")

The same filters work in the exercise below. Change the keyword, the period or the IIIF switch and run it again.

In [ ]:
# ============================================================
# EXERCISE: combine a keyword with filters
# ============================================================
SEARCH_KEYWORD = "ship"        # <-- try "portrait", "map", "horse", "cathedral"
YEARS          = (1800, 1899)  # <-- or None for any period
ONLY_IIIF      = True          # <-- True: only records that link a IIIF manifest
THEME          = None          # <-- or "art", "manuscripts", "photography", ...
# ============================================================

qf = ["TYPE:IMAGE"]
if YEARS:
    qf.append(f"YEAR:[{YEARS[0]} TO {YEARS[1]}]")
if ONLY_IIIF:
    qf.append("wr_dcterms_isReferencedBy:*")

results = search_europeana(query=SEARCH_KEYWORD, rows=12, qf=qf, theme=THEME, reusability="open")
if results and results.get("success"):
    print(f"{results['totalResults']:,} matching images (showing {len(results.get('items', []))})\n")
    cards = "".join(
        f"<figure style='margin:0;width:170px;font:11px sans-serif'><img src='{get_item_preview(it) or ''}' style='width:170px;height:170px;object-fit:cover;background:#eee'>"
        f"<figcaption><b>{html.escape(get_item_title(it)[:60])}</b><br>{html.escape(str(get_item_creator(it))[:40])} ({get_item_year(it)})<br>"
        f"<i>{html.escape((it.get('dataProvider') or [''])[0][:40])}</i></figcaption></figure>" for it in results["items"])
    display(HTML(f"<div style='display:flex;gap:12px;flex-wrap:wrap'>{cards}</div>"))
else:
    print("Search failed")

### 4.3 Export metadata as a dataset

The Search API pages through results with a *cursor*, so any number of records can be collected, not only the first 1,000. The cell below collects a few hundred records for a query into a table and saves it as CSV. Counting and sorting that table is then ordinary data analysis, outside any website. The same approach, scaled up, is how the shared dataset for the next notebook is built.

Note that results arrive in Europeana's default order, so a single institution can dominate (look at the counts printed below). A balanced dataset needs one query per institution.

In [ ]:
import csv
from collections import Counter

def first_of(v, default=""):
    return (v[0] if isinstance(v, list) and v else v) or default

def export_metadata(query, qf=None, theme=None, pages=3, rows=100):
    records, cursor = [], "*"
    for _ in range(pages):
        res = search_europeana(query=query, rows=rows, qf=qf, theme=theme, reusability="open", profile="standard", cursor=cursor)
        if not res or not res.get("items"):
            break
        for it in res["items"]:
            records.append({
                "id": it.get("id"), "title": get_item_title(it), "creator": get_item_creator(it), "year": get_item_year(it),
                "institution": first_of(it.get("dataProvider")), "provider": first_of(it.get("provider")),
                "country": first_of(it.get("country")), "type": it.get("type"), "rights": it.get("rights", [""])[0],
                "landing_page": first_of(it.get("edmIsShownAt")), "preview": get_item_preview(it) or "",
            })
        cursor = res.get("nextCursor")
        if not cursor:
            break
    return records

records = export_metadata("ship", qf=["TYPE:IMAGE", "wr_dcterms_isReferencedBy:*"])
out = DATA_DIR / "europeana" / "export_ship.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(records[0]))
    w.writeheader(); w.writerows(records)

print(f"{len(records)} records saved to {out.relative_to(PROJECT_ROOT)}\n")
print("Institutions:", Counter(r["institution"] for r in records).most_common(5))
print("Countries:   ", Counter(r["country"] for r in records).most_common(5))
print("Creators:    ", Counter(r["creator"] for r in records).most_common(5))

### 4.4 What is not available through the API

The API returns metadata and links, not the objects. The full-resolution images stay on the institution's server, which is where the IIIF technique from the first notebook comes in: with the `edmIsShownBy` link (or the manifest), a program can request a small version, a region, or a deep-zoom tile of each item. The metadata export above is therefore the first half of building an image dataset, and the IIIF image request is the second.

## 5. What one record contains

Each item in a search result is a Europeana record. A fresh sample of 20 art images is fetched here so that this section works whatever keyword was used above. The fields that matter for the flow in section 1 are:

| Field | Meaning | Where it appears in the diagram |
|---|---|---|
| `dataProvider` | the institution that holds the object | the platform at the bottom |
| `provider` | the aggregator or route | the middle row |
| `edmIsShownAt` | the object's page on the institution's website | link back to the source |
| `edmIsShownBy` | the media file on the institution's server | link back to the source |
| `edmPreview` | a small preview image served by Europeana | the only image Europeana itself serves |
| `rights` | the rights statement | |

In [ ]:
sample_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE"], theme="art", reusability="open")
sample = sample_results["items"]
item = sample[0]
for field in ["title", "dataProvider", "provider", "country", "type", "rights", "edmIsShownAt", "edmIsShownBy", "edmPreview"]:
    value = item.get(field)
    if isinstance(value, list):
        value = value[0]
    print(f"{field:<14} {value}")

### From a record to a IIIF manifest

Europeana publishes a manifest for each record at a predictable address, built from the record ID. That connects this notebook to the previous one: the manifest lists the canvases of the object, and its `service` entries, if present, are IIIF image services at the institution.

In [ ]:
def get_iiif_manifest_url(item):
    """
    Construct IIIF manifest URL from item ID.

    Europeana generates IIIF manifests on-the-fly for all items.
    Pattern: https://iiif.europeana.eu/presentation/{dataset}/{local_id}/manifest
    """
    item_id = item.get('id')
    if not item_id:
        return None
    parts = item_id.strip('/').split('/')
    if len(parts) >= 2:
        return f"https://iiif.europeana.eu/presentation/{parts[0]}/{parts[1]}/manifest"
    return None

In [ ]:
def manifest_has_image_service(item):
    # Fetch the item's IIIF manifest and report whether its images are served by an IIIF Image API service.
    url = get_iiif_manifest_url(item) + "?format=3"
    try:
        manifest = requests.get(url, timeout=30).json()
        body = manifest["items"][0]["items"][0]["items"][0]["body"]
        return bool(body.get("service"))
    except Exception:
        return None

with ThreadPoolExecutor(6) as ex:
    flags = list(ex.map(manifest_has_image_service, sample))

for it, flag in zip(sample, flags):
    mark = {True: "image service", False: "static image only", None: "no manifest"}[flag]
    print(f"{mark:<18} {get_item_title(it)[:50]:<52} {it.get('dataProvider', [''])[0][:35]}")
print(f"\n{sum(f is True for f in flags)} of {len(sample)} records have an IIIF image service.")

The same check on records selected with the IIIF filter from section 4.2 (`wr_dcterms_isReferencedBy:*`):

In [ ]:
iiif_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE", "wr_dcterms_isReferencedBy:*"], theme="art", reusability="open")
iiif_sample = iiif_results["items"]
with ThreadPoolExecutor(6) as ex:
    iiif_flags = list(ex.map(manifest_has_image_service, iiif_sample))
print(f"{sum(f is True for f in iiif_flags)} of {len(iiif_sample)} records selected with the IIIF filter have an image service.")

Not every record has one, and the IIIF filter is only reliable in one direction: records selected with it do have an image service, but records without the filter can have one too (the Wellcome record above), so the manifest check on each record is the reliable test. Where a manifest exists, the techniques from the previous notebook apply: region requests, deep zoom and annotations. Where an institution only supplies a plain image file, the record is still searchable but the pixels cannot be requested in parts.

## Summary

- Europeana is an **aggregator**: it harvests metadata (as EDM) from institutions, directly, through national or domain aggregators, or through projects. The full-resolution media stay with the institutions.
- IIIF and aggregation are complementary. Aggregation helps with *finding* objects across institutions, and IIIF helps with *viewing and using* them once found.
- The Search API can return items or counts. Counting by `DATA_PROVIDER`, `PROVIDER`, `COUNTRY` and `TYPE` gives an overview of who contributes what.
- The API is most useful for what a search box does not do: distributions over a field, stacked and range filters, and exporting metadata as a table.
- Contributions are very uneven, and the `TYPE` field describes media, not genre.

### Resources

- Europeana APIs: <https://pro.europeana.eu/page/apis>
- Search API reference: <https://pro.europeana.eu/page/search>
- Europeana Data Model (EDM): <https://pro.europeana.eu/page/edm-documentation>

## Next

The next notebook works with a shared dataset: small images from many of these institutions, collected through exactly the requests shown here and through IIIF, and their precomputed CLIP embeddings. You will search that collection with your own text queries.